# Task A Run 35 -- exact holdout twin of Run 33

This notebook validates the current Task A Run 33 recipe before relying on its
full-data test submission. It uses the fixed 85%/15% split (`815110ff24`):

1. TAPT on the 6,114 training comments plus the external Kannada corpus only.
2. Four TAPT-initialised Gemma classifiers, seeds 42--45, 3 epochs, short prompt and
   learning rate `1e-4`, matching Run 33.
3. A matched no-TAPT control for the same seeds, reusing attached healthy outputs when
   available.
4. Paired-bootstrap comparison of the four-model TAPT and control averages on the
   1,079-row holdout.

This is evaluation-only: it does not read test comments and does not create a ZIP.
TAPT runs before any classifier jobs. If TAPT fails, the control is not trained.

In [ ]:
import json, os, pathlib, subprocess, sys, time
from fnmatch import fnmatch
T_START = time.time()
HARD_STOP = T_START + 11 * 3600
REPO = 'https://github.com/robinpnalex/Hastika-ICON2026.git'
BRANCH = 'task-b'
WORK = '/tmp/hastika'

def git(*args):
    return subprocess.run(['git', '-C', WORK, *args], check=True, capture_output=True, text=True).stdout.strip()

remote = subprocess.run(['git', 'ls-remote', '--exit-code', REPO, f'refs/heads/{BRANCH}'],
                         check=True, capture_output=True, text=True, timeout=60).stdout.split()[0]
if os.path.isdir(WORK + '/.git'):
    git('fetch', '-q', '--depth', '1', 'origin', BRANCH)
    git('reset', '-q', '--hard', 'FETCH_HEAD')
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, '--depth', '1', REPO, WORK], check=True)
head = git('rev-parse', 'HEAD')
assert head == remote, f'clone is at {head[:8]} but {BRANCH} is at {remote[:8]}'
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))
print('repo at', git('log', '-1', '--oneline'))
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf \"transformers>=4.45,<6\"', shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
assert torch.cuda.is_available(), 'select GPU T4 x2'
N_GPU = torch.cuda.device_count()
print('gpus:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])
OUT = pathlib.Path('/kaggle/working/a35_outputs')
RUNS, LOGS = OUT / 'runs', OUT / 'logs'
for p in (RUNS, LOGS):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = '/tmp/hf'
os.environ['HF_HUB_CACHE'] = HF_CACHE
from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import fingerprint, paired_bootstrap


In [ ]:
LLM_PY = '/tmp/llmenv/bin/python'
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', '/tmp/llmenv'], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q \"transformers>=5.12,<6\" \"peft>=0.17\" \"bitsandbytes>=0.46\" \"accelerate>=1.0\" emoji ftfy sentencepiece protobuf', shell=True, check=True)
subprocess.run([LLM_PY, '-c', "import transformers, peft, bitsandbytes; print('LLM env:', transformers.__version__, peft.__version__)"], check=True)
MODEL = 'google/gemma-4-12B'
TASK = 'a'
CLASSES = ['Non-Hate', 'Hate']
LABEL_COL = 'Label'
SPLIT = pathlib.Path(WORK) / 'data/derived/task_a_combined_holdout'
TRAIN = SPLIT / 'train.csv'
HOLDOUT = SPLIT / 'holdout.csv'
EXTERNAL = pathlib.Path(WORK) / 'data/external/offenseval_kn.csv'
train = pd.read_csv(TRAIN)
holdout = pd.read_csv(HOLDOUT)
assert (len(train), len(holdout)) == (6114, 1079), (len(train), len(holdout))
assert fingerprint(holdout['id']) == '815110ff24'
y_ho = holdout[LABEL_COL].map(CLASSES.index).to_numpy()
print(f'train {len(train)}, holdout {len(holdout)} ({fingerprint(holdout["id"])}), full corpus {len(train) + len(holdout)}')

def download():
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL, cache_dir=HF_CACHE, token=os.environ.get('HF_TOKEN') or None,
                      allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt', 'tokenizer*', '*.jinja'])
download()


In [ ]:
SEEDS = [42, 43, 44, 45]
TAPT_DIR = RUNS / 'tapt_train'

def tapt_job(name, bs=1, grad_accum=16, max_len=128, logit_chunk=16):
    texts = ' '.join(str(p) for p in (TRAIN, EXTERNAL))
    cmd = (f'{LLM_PY} -u -m hastika.task_b.llm_tapt --model {MODEL} '
           f'--texts {texts} --out {TAPT_DIR} --cache {HF_CACHE} '
           f'--bs {bs} --grad-accum {grad_accum} --max-len {max_len} --logit-chunk {logit_chunk}')
    return {'name': name, 'log': str(LOGS / f'{name}.log'), 'cmd': cmd}

def classifier_job(kind, seed, name=None, init=False, extra=''):
    name = name or f'{kind}_s{seed}'
    adapter = f'--init-adapter {TAPT_DIR}' if init else ''
    cmd = (f'{LLM_PY} -u -m hastika.task_b.llm_classifier --task {TASK} --model {MODEL} '
           f'--train {TRAIN} --eval {HOLDOUT} --predict {HOLDOUT} --out {RUNS / name} '
           f'--seed {seed} --epochs 3 --prompt short --lr 1e-4 --class-weight balanced '
           f'--cache {HF_CACHE} {adapter} {extra}')
    return {'name': name, 'log': str(LOGS / f'{name}.log'), 'cmd': cmd}

def load_probs(run_dir, csv=HOLDOUT):
    run_dir, csv = pathlib.Path(run_dir), pathlib.Path(csv)
    p, ids = run_dir / f'{csv.stem}_probs.npy', run_dir / f'{csv.stem}_ids.csv'
    if not p.exists() or not ids.exists():
        return None
    if pd.read_csv(ids)['id'].tolist() != pd.read_csv(csv)['id'].tolist():
        return None
    probs = np.load(p)
    share = np.bincount(probs.argmax(1), minlength=len(CLASSES)) / len(probs)
    return None if share.max() > 0.9 else probs

def find_prior(pattern):
    hits = []
    for root, _, files in os.walk('/kaggle/input', followlinks=True):
        for f in files:
            p = os.path.join(root, f)
            if fnmatch(p, pattern):
                hits.append(pathlib.Path(p))
    return sorted(hits)

def run_tapt():
    first = tapt_job('tapt_train')
    codes = run_queue([first], N_GPU, raise_on_fail=False, stop_at=HARD_STOP)
    if codes.get('tapt_train', 1) == 0 and (TAPT_DIR / 'adapter_config.json').exists():
        return codes
    print('TAPT failed; retrying at 64 tokens')
    retry = tapt_job('tapt_retry', bs=1, grad_accum=16, max_len=64, logit_chunk=8)
    retry_codes = run_queue([retry], N_GPU, raise_on_fail=False, stop_at=HARD_STOP)
    assert retry_codes.get('tapt_retry', 1) == 0 and (TAPT_DIR / 'adapter_config.json').exists(), 'TAPT failed twice; no classifier jobs were started'
    return {**codes, **retry_codes}


In [ ]:
tapt_codes = run_tapt()
print('TAPT exit codes:', tapt_codes)

# Reuse identical holdout probabilities from attached prior runs where possible.
probs = {'tapt': {}, 'base': {}}
jobs = []
for seed in SEEDS:
    for pat in (f'*/runs/tapt_s{seed}/holdout_probs.npy',):
        got = [load_probs(h.parent) for h in find_prior(pat)]
        got = [p for p in got if p is not None]
        if got:
            probs['tapt'][seed] = got[0]
            break
    for pat in (f'*/runs/base_s{seed}/holdout_probs.npy', f'*/runs/ho_s{seed}/holdout_probs.npy'):
        got = [load_probs(h.parent) for h in find_prior(pat)]
        got = [p for p in got if p is not None]
        if got:
            probs['base'][seed] = got[0]
            break
    if seed not in probs['tapt']:
        jobs.append(classifier_job('tapt', seed, init=True))
    if seed not in probs['base']:
        jobs.append(classifier_job('base', seed, init=False))
print('classifier jobs:', [j['name'] for j in jobs])

registry = {j['name']: j for j in jobs}
retried = set()
def classifier_done(name, code):
    if not code or name in retried:
        return []
    job = registry[name]
    log = pathlib.Path(job['log']).read_text(errors='replace') if pathlib.Path(job['log']).exists() else ''
    import re
    seed_match = re.search(r'--seed (\d+)', job['cmd'])
    if code == 4 and seed_match:
        seed = int(seed_match.group(1)) + 1000
        retry = classifier_job('tapt' if name.startswith('tapt_') else 'base', seed, name=name, init=name.startswith('tapt_'))
        print(f'{name} collapsed; retrying with seed {seed}')
    elif 'OutOfMemoryError' in log:
        seed = int(seed_match.group(1))
        retry = classifier_job('tapt' if name.startswith('tapt_') else 'base', seed, name=name, init=name.startswith('tapt_'), extra='--bs 4 --grad-accum 4 --eval-bs 8')
        print(f'{name} OOM; retrying with smaller classifier batch')
    else:
        return []
    retried.add(name)
    registry[name] = retry
    return [retry]

classifier_codes = run_queue(jobs, N_GPU, raise_on_fail=False, on_done=classifier_done, stop_at=HARD_STOP) if jobs else {}
for kind in ('tapt', 'base'):
    for seed in SEEDS:
        p = load_probs(RUNS / f'{kind}_s{seed}')
        if p is not None:
            probs[kind][seed] = p
print('classifier exit codes:', classifier_codes)
print('elapsed:', f'{(time.time() - T_START) / 3600:.1f} h')


In [ ]:
def report(y, p, name):
    pred = p.argmax(1)
    macro = float(f1_score(y, pred, average='macro'))
    acc = float(accuracy_score(y, pred))
    print(f'{name:34s} macro-F1 {macro:.4f}  accuracy {acc:.4f}')
    return {'macro_f1': macro, 'accuracy': acc}

scores, averages = {}, {}
for kind in ('base', 'tapt'):
    if probs[kind]:
        averages[kind] = np.mean(list(probs[kind].values()), axis=0)
        scores[kind] = report(y_ho, averages[kind], f'{kind} ({len(probs[kind])}/{len(SEEDS)} seeds)')

bootstrap = {}
if set(probs['tapt']) == set(SEEDS) and set(probs['base']) == set(SEEDS):
    bootstrap['tapt_minus_base'] = paired_bootstrap(y_ho, averages['tapt'], averages['base'])
    b = bootstrap['tapt_minus_base']
    print(f"TAPT - base {b['diff']:+.4f}  95% CI [{b['ci95'][0]:+.4f}, {b['ci95'][1]:+.4f}]  P(better) {b['p_better']:.2f}")
else:
    print('No complete four-seed paired comparison; inspect the logs and attached outputs.')

json.dump({'commit': head, 'holdout_fingerprint': fingerprint(holdout['id']),
           'scores': scores, 'bootstrap': bootstrap,
           'members': {k: sorted(v) for k, v in probs.items()},
           'tapt_exit_codes': tapt_codes, 'classifier_exit_codes': classifier_codes},
          open(OUT / 'result.json', 'w'), indent=2)
print('result saved to', OUT / 'result.json')
